# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 머신러닝 모델이 학습 데이터에 너무 잘 맞춰져서, 새로운 데이터(테스트 데이터)에서는 성능이 떨어지는 현상을 말합니다. 즉, 모델이 학습 데이터의 노이즈나 특이점까지 학습하게 되어 일반화 능력이 떨어지는 경우입니다.

과적합의 주요 원인은 다음과 같습니다:

1. **모델의 복잡성**: 너무 복잡한 모델(예: 많은 파라미터를 가진 딥러닝 모델)은 학습 데이터의 세부사항을 과도하게 학습할 수 있습니다.
2. **학습 데이터의 부족**: 학습 데이터가 적거나 다양성이 부족할 경우, 모델이 특정 데이터에만 최적화될 수 있습니다.
3. **노이즈**: 학습 데이터에 포함된 노이즈나 오류가 모델에 영향을 미쳐, 이를 학습하게 되는 경우입니다.

과적합을 방지하기 위한 방법으로는 다음과 같은 것들이 있습니다:

- **교차 검증**: 데이터를 여러 부분으로 나누어 모델을 평가하여 일반화 성능을 확인합니다.
- **정규화**: L1, L2 정규화와 같은 기법을 사용하여 모델의 복잡성을 줄입니다.
- **드롭아웃**: 신경망에서 일부 뉴런을 무작위로 생략하여 과적합을 방지합니다.
- **조기 종료**: 검증 데이터의 성능이 더 이상 개선되지 않을 때 학습을 중단합니다.
- **데이터 증강**: 학습 데이터를 인위적으로 늘려 모델이 다양한 패턴을 학습하도록 합니다.

이러한 방법들을 통해 과적합을 줄이고, 모델의 일반화 능력을 향상시킬 수 있습니다.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [6]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

홍길동님이라고 하셨습니다! 맞나요?


In [8]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이라고 하셨습니다! 맞나요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이라고 하셨습니다! 맞나요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.
